# 10 Minutes in the middle

In [ ]:
from pathlib import Path
import subprocess
import json
from faster_whisper import WhisperModel

In [ ]:
def get_video_duration(video_path: str, ffprobe_path: str = "ffprobe") -> float:
    video_file = Path(video_path)

    if not video_file.exists():
        raise FileNotFoundError(f"video file not found: {video_file}")

    cmd = [
        ffprobe_path,
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "json",
        str(video_file)
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("ffprobe failed while reading video duration")

    data = json.loads(result.stdout)
    return float(data["format"]["duration"])

In [ ]:
def detect_language_from_middle_sample(
    video_path: str,
    sample_audio_path: str = "middle_language_sample.wav",
    ffmpeg_path: str = "ffmpeg",
    ffprobe_path: str = "ffprobe",
    model_size: str = "medium",
    device: str = "cpu",
    compute_type: str = "int8",
    sample_seconds: int = 600
):
    """
    Extract 10 minutes from the middle of a video and detect the language.
    """

    video_file = Path(video_path)
    sample_file = Path(sample_audio_path)

    if not video_file.exists():
        raise FileNotFoundError(f"video file not found: {video_file}")

    duration = get_video_duration(video_path, ffprobe_path=ffprobe_path)

    if duration <= sample_seconds:
        start_time = 0
    else:
        start_time = max(0, (duration / 2) - (sample_seconds / 2))

    print(f"video duration: {duration:.2f} seconds")
    print(f"sampling from: {start_time:.2f} to {start_time + sample_seconds:.2f} seconds")

    cmd = [
        ffmpeg_path,
        "-y",
        "-ss", str(start_time),
        "-i", str(video_file),
        "-t", str(sample_seconds),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        str(sample_file)
    ]

    print("extracting middle sample audio...")
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("ffmpeg failed while extracting sample audio")

    print(f"sample created: {sample_file}")

    model = WhisperModel(model_size, device=device, compute_type=compute_type)

    segments, info = model.transcribe(
        str(sample_file),
        language=None,
        beam_size=5
    )

    preview_segments = list(segments)

    print(f"detected language: {info.language}")
    print(f"language probability: {info.language_probability:.3f}")

    print("\npreview of detected speech:")
    for i, segment in enumerate(preview_segments[:10]):
        print(f"[{segment.start:.2f} -> {segment.end:.2f}] {segment.text}")

    return info.language, info.language_probability, preview_segments

In [ ]:
lang, prob, segs = detect_language_from_middle_sample(
    video_path="your_video.mp4",
    ffmpeg_path="/opt/homebrew/opt/ffmpeg-full/bin/ffmpeg",
    ffprobe_path="/opt/homebrew/opt/ffmpeg-full/bin/ffprobe",
    # model_size="large-v3"
    model_size="medium"
)

In [ ]:
from pathlib import Path

wav_file = Path("middle_language_sample.wav")

if wav_file.exists():
    wav_file.unlink()
    print(f"deleted: {wav_file}")
else:
    print(f"file not found: {wav_file}")

# 10 minutes from the start

In [ ]:
from pathlib import Path
import subprocess
from faster_whisper import WhisperModel


def detect_language_from_start_sample(
    video_path: str,
    sample_audio_path: str = "start_language_sample.wav",
    ffmpeg_path: str = "ffmpeg",
    model_size: str = "medium",
    device: str = "cpu",
    compute_type: str = "int8",
    sample_seconds: int = 600
):
    """
    Extract the first 10 minutes of a video and detect the language.
    """

    video_file = Path(video_path)
    sample_file = Path(sample_audio_path)

    if not video_file.exists():
        raise FileNotFoundError(f"video file not found: {video_file}")

    cmd = [
        ffmpeg_path,
        "-y",
        "-i", str(video_file),
        "-t", str(sample_seconds),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        str(sample_file)
    ]

    print("extracting start sample audio...")
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("ffmpeg failed while extracting sample audio")

    print(f"sample created: {sample_file}")

    model = WhisperModel(model_size, device=device, compute_type=compute_type)

    segments, info = model.transcribe(
        str(sample_file),
        language=None,
        beam_size=5
    )

    preview_segments = list(segments)

    print(f"detected language: {info.language}")
    print(f"language probability: {info.language_probability:.3f}")

    print("\npreview of detected speech:")
    for segment in preview_segments[:10]:
        print(f"[{segment.start:.2f} -> {segment.end:.2f}] {segment.text}")

    return info.language, info.language_probability, preview_segments

In [ ]:
lang, prob, segs = detect_language_from_start_sample(
    video_path="your_video.mp4",
    ffmpeg_path="/opt/homebrew/opt/ffmpeg-full/bin/ffmpeg",
    model_size="medium"
)

In [ ]:
from pathlib import Path

wav_file = Path("start_language_sample.wav")

if wav_file.exists():
    wav_file.unlink()
    print(f"deleted: {wav_file}")
else:
    print(f"file not found: {wav_file}")